In [1]:
# --- 1. LIBRARIES ---
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning & Scaling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (classification_report, confusion_matrix, 
                             accuracy_score, roc_curve, auc)

# Deep Learning (TensorFlow/Keras)
import tensorflow as tf
from tensorflow.keras import Model, Input
from tensorflow.keras.layers import (Dense, Conv1D, BatchNormalization, 
                                     concatenate, Flatten, Dropout, Attention, 
                                     Reshape, GlobalAveragePooling1D, Multiply, MaxPooling1D)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

C:\Users\Cloud-2\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [2]:

# --- 2. REPRODUCIBILITY (The "Same Result" Rule) ---
def set_seeds(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    tf.keras.utils.set_random_seed(seed)
    print(f"Environment seeds locked at {seed}")

set_seeds(42)

Environment seeds locked at 42


In [3]:
# Load the specific binary dataset
df = pd.read_csv("dataset_name.csv", engine="pyarrow")

# Verify the load by checking the total rows and columns
print(f"Dataset successfully loaded.")
print(f"Total Rows: {df.shape[0]}")
print(f"Total Columns: {df.shape[1]}")

Dataset successfully loaded.
Total Rows: 100309665
Total Columns: 84


In [4]:
# 1. Strip any leading/trailing whitespace and convert to lowercase
df.columns = df.columns.str.strip().str.lower()

# 2. Replace spaces, slashes, and dots with underscores for easy coding
df.columns = df.columns.str.replace(' ', '_', regex=False)
df.columns = df.columns.str.replace('/', '_', regex=False)
df.columns = df.columns.str.replace('.', '_', regex=False)

# 3. Print the first 10 columns to verify the change
print("First 10 standardized columns:")
print(df.columns[:10].tolist())

# 4. Find the exact name of your Label column
label_col = [col for col in df.columns if 'label' in col]
print(f"\nLabel column found: {label_col}")

First 10 standardized columns:
['flow_id', 'src_ip', 'src_port', 'dst_ip', 'dst_port', 'protocol', 'timestamp', 'flow_duration', 'tot_fwd_pkts', 'tot_bwd_pkts']

Label column found: ['label']


In [5]:
# List of columns that are not useful for training features
identifiers = ['flow_id', 'src_ip', 'dst_ip', 'timestamp']

# Drop them only if they exist in the current dataframe
df.drop(columns=[col for col in identifiers if col in df.columns], inplace=True)

print(f"Identifiers removed.")
print(f"Remaining columns: {df.shape[1]}")

Identifiers removed.
Remaining columns: 80


In [6]:
df.shape

(100309665, 80)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100309665 entries, 0 to 100309664
Data columns (total 80 columns):
 #   Column             Dtype  
---  ------             -----  
 0   src_port           int64  
 1   dst_port           int64  
 2   protocol           int64  
 3   flow_duration      int64  
 4   tot_fwd_pkts       int64  
 5   tot_bwd_pkts       int64  
 6   totlen_fwd_pkts    float64
 7   totlen_bwd_pkts    float64
 8   fwd_pkt_len_max    float64
 9   fwd_pkt_len_min    float64
 10  fwd_pkt_len_mean   float64
 11  fwd_pkt_len_std    float64
 12  bwd_pkt_len_max    float64
 13  bwd_pkt_len_min    float64
 14  bwd_pkt_len_mean   float64
 15  bwd_pkt_len_std    float64
 16  flow_byts_s        float64
 17  flow_pkts_s        float64
 18  flow_iat_mean      float64
 19  flow_iat_std       float64
 20  flow_iat_max       float64
 21  flow_iat_min       float64
 22  fwd_iat_tot        float64
 23  fwd_iat_mean       float64
 24  fwd_iat_std        float64
 25  fwd_iat_max   

In [8]:
df.drop_duplicates(inplace=True)

In [9]:
df['label'].value_counts()

label
1    90694000
0     7331139
Name: count, dtype: int64

In [10]:
import joblib
import numpy as np

imputer_bundle = joblib.load("regression_imputer.pkl")

imputer_models = imputer_bundle['models']
COLS_WONA = imputer_bundle['cols_wona']
COLS_WNA = imputer_bundle['cols_wna']


In [11]:
# Replace Inf with NaN (same as training)
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [12]:
print("Remaining NaNs:", df.isna().sum().sum())

Remaining NaNs: 2788764


In [13]:
import gc

chunk_size = 5_000_000

for target_col, model in imputer_models.items():
    if target_col not in df.columns:
        continue  # safety check

    # Get indices of missing values
    predict_idx = df.index[df[target_col].isna()]
    
    if len(predict_idx) > 0:
        print(f"Applying predictions for: {target_col}")
        
        # Predict and fill in chunks
        for i in range(0, len(predict_idx), chunk_size):
            idx_chunk = predict_idx[i : i + chunk_size]
            X_pred_chunk = df.loc[idx_chunk, COLS_WONA].values
            
            preds = model.predict(X_pred_chunk)
            df.loc[idx_chunk, target_col] = preds
            
            del X_pred_chunk, preds
            
    # Force RAM cleanup after each column
    gc.collect()

Applying predictions for: flow_byts_s


C:\Users\Cloud-2\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


Applying predictions for: flow_pkts_s


C:\Users\Cloud-2\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [14]:
print("Remaining NaNs:", df.isna().sum().sum())


Remaining NaNs: 0


In [15]:
# 1. Check for any columns that are not integers or floats (excluding 'label')
non_numeric = df.drop(columns=['label']).select_dtypes(exclude=[np.number]).columns.tolist()
print(f"Non-numeric columns found (excluding label): {non_numeric}")

# 2. Check for Infinite values (common in this dataset)
inf_count = np.isinf(df.select_dtypes(include=[np.number])).values.sum()
print(f"Total Infinite values: {inf_count}")

# 3. Check for Null/NaN values
nan_count = df.isnull().sum().sum()
print(f"Total NaN values: {nan_count}")

Non-numeric columns found (excluding label): []
Total Infinite values: 0
Total NaN values: 0


In [16]:
# 1. Check unique values in the label column
print("Unique labels in dataset:")
print(df['label'].unique())

# 2. Check the count of each label to see if it is imbalanced
print("\nLabel counts:")
print(df['label'].value_counts())

# 3. Check the data type of the label
print(f"\nLabel data type: {df['label'].dtype}")

Unique labels in dataset:
[1 0]

Label counts:
label
1    90694000
0     7331139
Name: count, dtype: int64

Label data type: int64


In [17]:
import gc
import joblib

# 1. Define Features (X) and Target (y)
X = df.drop(columns=['label'])
y = df['label']

# CLEAR RAM: Original dataframe is no longer needed
del df
gc.collect()

# 2. Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# CLEAR RAM: X is no longer needed
del X
gc.collect()

print(f"Training set: {X_train.shape[0]} rows")
print(f"Testing set:  {X_test.shape[0]} rows")

scaler = joblib.load("scaler.pkl")
print("Scaler loaded successfully")

X_train_scaled = scaler.transform(X_train)
# CLEAR RAM: Delete unscaled X_train
del X_train
gc.collect()

X_test_scaled = scaler.transform(X_test)
# CLEAR RAM: Delete unscaled X_test
del X_test
gc.collect()

print(f"Scaling Complete. RAM cleaned.")
print(f"Train dtype: {X_train_scaled.dtype}")
print(f"Train shape: {X_train_scaled.shape}")

Training set: 78420111 rows
Testing set:  19605028 rows
Scaler loaded successfully
Scaling Complete. RAM cleaned.
Train dtype: float64
Train shape: (78420111, 79)


In [18]:
from tensorflow.keras.models import load_model

# Load encoder (for feature extraction)
encoder_only = load_model("AE_Encoder_Extractor.keras")

# (Optional) Load full autoencoder if you need reconstruction
autoencoder = load_model("AE_Stage1_Full.keras")

print("Models loaded successfully")



Models loaded successfully


In [19]:
# 1. Do reconstruction check FIRST while scaled data is still in RAM
X_train_recon = autoencoder.predict(X_train_scaled[:1000])
recon_error = np.mean(np.abs(X_train_scaled[:1000] - X_train_recon))
print("Reconstruction MAE:", recon_error)

# 2. Extract 32 latent features for training data
X_train_encoded = encoder_only.predict(X_train_scaled, batch_size=1024)

# CLEAR RAM: Delete the massive scaled training data!
import gc
del X_train_scaled
gc.collect()

# 3. Extract 32 latent features for testing data
X_test_encoded = encoder_only.predict(X_test_scaled, batch_size=1024)

# CLEAR RAM: Delete the scaled testing data!
del X_test_scaled
gc.collect()

print("Encoded shapes ready for Logistic Regression:")
print(f"X_train_encoded: {X_train_encoded.shape}")  # Should be (samples, 32)
print(f"X_test_encoded: {X_test_encoded.shape}")    # Should be (samples, 32)gf

 1/32 ━━━━━━━━━━━━━━━━━━━━ 13s 420ms/step

C:\Users\Cloud-2\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\ops\nn.py:947: UserWarning: You are using a softmax over axis -1 of a tensor of shape (32, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
Reconstruction MAE:

C:\Users\Cloud-2\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\ops\nn.py:947: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


 0.0009092664698261463
   73/76583 ━━━━━━━━━━━━━━━━━━━━ 2:13 2ms/step

C:\Users\Cloud-2\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\ops\nn.py:947: UserWarning: You are using a softmax over axis -1 of a tensor of shape (1024, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(


76583/76583 ━━━━━━━━━━━━━━━━━━━━ 141s 2ms/step
19146/19146 ━━━━━━━━━━━━━━━━━━━━ 40s 2ms/step
Encoded shapes:
(78420111, 32)
(19605028, 32)
CNN-ready shapes:
(78420111, 32, 1)
(19605028, 32, 1)


In [ ]:
# --- LOGISTIC REGRESSION CLASSIFIER (High-Memory Server Optimized) ---
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_curve, auc, matthews_corrcoef
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 1. No Reshaping Needed!
# LR expects (Samples, Features). X_train_encoded is already (Samples, 32).
print(f"LR Training Shape: {X_train_encoded.shape}")
print(f"LR Testing Shape:  {X_test_encoded.shape}")

# 2. Initialize the Standard Logistic Regression model
# With 128GB RAM, the standard 'lbfgs' solver is perfectly safe and highly accurate.
lr_model = LogisticRegression(
    max_iter=2000,         # Increased iterations to ensure convergence on large data
    n_jobs=-1,             # Engages all Xeon CPU cores for parallel processing
    random_state=42,
    verbose=1              # Prints progress during training
)

# 3. Train the Model
print("Starting Standard Logistic Regression Training...")
lr_model.fit(X_train_encoded, y_train)
print("Logistic Regression Training Complete.")

# 4. Save the Model
joblib.dump(lr_model, "LR_Classifier_Standard.pkl")
print("LR model saved successfully as LR_Classifier_Standard.pkl")

# --- EVALUATION & METRICS ---

# 1. Predict Classes and Probabilities
y_pred = lr_model.predict(X_test_encoded)
y_pred_probs = lr_model.predict_proba(X_test_encoded)

# 2. Print Core Metrics
print(f"\nFinal Test Accuracy: {accuracy_score(y_test, y_pred):.5f}")
mcc = matthews_corrcoef(y_test, y_pred)
print(f"Matthews Correlation Coefficient (MCC): {mcc:.5f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['BENIGN', 'ATTACK']))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# --- VISUALIZATION ---

# 1. Setup the figure for Matrices
plt.figure(figsize=(16, 6))

# Plot 1: Confusion Matrix Heatmap
plt.subplot(1, 2, 1)
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['BENIGN', 'ATTACK'], 
            yticklabels=['BENIGN', 'ATTACK'])
plt.title('Confusion Matrix (Standard Logistic Regression)', fontsize=14)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')

# Plot 2: Classification Report Heatmap
plt.subplot(1, 2, 2)
report_dict = classification_report(y_test, y_pred, 
                                    target_names=['BENIGN', 'ATTACK'], 
                                    output_dict=True)
report_df = pd.DataFrame(report_dict).iloc[:-1, :2].T 
sns.heatmap(report_df, annot=True, cmap='RdYlGn', cbar=True, annot_kws={"size": 12})
plt.title('Classification Metrics (Precision, Recall, F1)', fontsize=14)

plt.tight_layout()
plt.show()

# 2. Plot ROC Curve
# Get the probability for the "Attack" class (class 1)
y_probs_attack = y_pred_probs[:, 1]
fpr, tpr, thresholds = roc_curve(y_test, y_probs_attack)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=14)
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=14)
plt.title('Receiver Operating Characteristic (ROC) - AE + Logistic Regression', fontsize=16)
plt.legend(loc="lower right", fontsize=12)
plt.grid(alpha=0.3)
plt.show()

print(f"Final AUC Score: {roc_auc:.5f}")